# Notebook 1 — Analyse Exploratoire des Donnees (EDA)
## Projet : Return/Refund Propensity Classification — Olist E-Commerce

**Objectif de ce notebook :** Explorer le dataset Olist, comprendre la structure des donnees,
identifier les problemes de qualite et degager des insights utiles avant la modelisation.

---
### Plan
1. Chargement des donnees
2. Vue d'ensemble du dataset
3. Analyse des valeurs manquantes
4. Distribution de la variable cible proxy
5. Analyse des variables numeriques
6. Analyse des variables categorielles
7. Analyse temporelle
8. Correlations
9. Conclusions de l'EDA

## 0. Imports & Configuration

In [ ]:
# --- Librairies standard ---
import pandas as pd
import numpy as np
import warnings
import sys
import os

# --- Visualisation ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- Configuration globale ---
warnings.filterwarnings('ignore')          # Masquer les warnings non-critiques
pd.set_option('display.max_columns', 50)   # Afficher jusqu'à 50 colonnes
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 décimales pour les floats

# Style des graphiques
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

# Ajouter le dossier racine au PATH pour pouvoir importer src/
sys.path.append(os.path.abspath('..'))

print('✅ Imports réussis')

---
## 1. Chargement des Données

On utilise le module `src/data_loader.py` qui charge et fusionne automatiquement les 9 tables CSV.

In [ ]:
from src.data_loader import load_data

# Charge et fusionne les 9 fichiers CSV en un seul DataFrame
df = load_data(data_dir='../data')

# Aperçu rapide
print(f'\nShape du DataFrame : {df.shape}')

---
## 2. Vue d'Ensemble du Dataset

In [ ]:
# Afficher les 5 premières lignes
df.head()

In [ ]:
# Informations sur les types de données et les valeurs non-nulles
df.info()

In [ ]:
# Statistiques descriptives pour les colonnes numériques
# count, mean, std, min, 25%, 50%, 75%, max
df.describe()

In [ ]:
# Distribution des statuts de commande
print('Distribution des statuts de commande :')
print(df['order_status'].value_counts())

# Pour la target proxy return/refund, on conserve delivered + canceled + unavailable

---
## 3. Analyse des Valeurs Manquantes

Identifier les NaN est une étape critique : ils peuvent biaiser les modèles si mal gérés.

In [ ]:
# Calcul du taux de valeurs manquantes par colonne
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'nb_manquants': missing, 'pct_manquants': missing_pct})
missing_df = missing_df[missing_df['nb_manquants'] > 0].sort_values('pct_manquants', ascending=False)

print('Colonnes avec des valeurs manquantes :')
print(missing_df)

In [ ]:
# Visualisation des valeurs manquantes
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['pct_manquants'].plot(kind='barh', ax=ax, color='salmon')
    ax.set_xlabel('% de valeurs manquantes')
    ax.set_title('Taux de Valeurs Manquantes par Colonne', fontweight='bold')
    ax.axvline(x=30, color='red', linestyle='--', label='Seuil 30%')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Aucune valeur manquante détectée dans le DataFrame fusionné.')

---
## 4. Distribution de la Variable Cible Proxy (Return/Refund Risk)

Comprendre la distribution de la cible proxy est essentiel pour :
- Quantifier le desequilibre des classes
- Valider les hypotheses de modeling
- Orienter les strategies de mitigation

In [ ]:
# Construire une cible proxy return/refund risk
df_scope = df[df['order_status'].isin(['delivered', 'canceled', 'unavailable'])].copy()
df_scope['is_return_refund_risk'] = (
    df_scope['order_status'].isin(['canceled', 'unavailable']) |
    (df_scope['review_score'].fillna(5) <= 2)
).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Distribution par statut
status_counts = df_scope['order_status'].value_counts()
axes[0].bar(status_counts.index, status_counts.values, color=['#2ecc71', '#e67e22', '#e74c3c'])
axes[0].set_xlabel('order_status')
axes[0].set_ylabel('Nombre de commandes')
axes[0].set_title('Distribution des Statuts de Commande', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

# Graphique 2 : Distribution binaire du risque
risk_counts = df_scope['is_return_refund_risk'].value_counts().sort_index()
labels = ['Low Risk (0)', 'High Risk (1)']
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(risk_counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Repartition du risque Return/Refund', fontweight='bold')

plt.suptitle('Analyse de la Variable Cible Proxy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nDesequilibre des classes : low risk={risk_counts.get(0,0):,} | high risk={risk_counts.get(1,0):,}")
print(f"Ratio low/high : {risk_counts.get(0,1)/max(risk_counts.get(1,1),1):.1f}:1")

# Conserver un DataFrame de reference pour les analyses suivantes
df_delivered = df_scope.copy()

---
## 5. Analyse des Variables Numériques

Étude des distributions et des outliers pour les variables quantitatives clés.

In [ ]:
# Colonnes numériques à analyser
numeric_cols = ['total_price', 'total_freight', 'payment_value',
                 'item_count', 'product_weight_g', 'payment_installments']

# Filtrer uniquement les colonnes existantes
numeric_cols = [c for c in numeric_cols if c in df_delivered.columns]

# Histogrammes avec boîtes à moustaches
fig, axes = plt.subplots(2, len(numeric_cols)//2, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = df_delivered[col].dropna()
    axes[i].hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution : {col}', fontsize=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Fréquence')
    # Ajouter médiane et moyenne pour comparer
    axes[i].axvline(data.median(), color='orange', linestyle='--', label=f'Médiane: {data.median():.1f}')
    axes[i].axvline(data.mean(),   color='red',    linestyle='-',  label=f'Moyenne: {data.mean():.1f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Distributions des Variables Numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots pour détecter les outliers
# Un outlier est une valeur anormalement éloignée du reste (au-delà de 1.5 × IQR)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

cols_to_plot = ['total_price', 'total_freight', 'payment_value']
cols_to_plot = [c for c in cols_to_plot if c in df_delivered.columns]

for i, col in enumerate(cols_to_plot):
    df_delivered.boxplot(column=col, ax=axes[i])
    axes[i].set_title(f'Boxplot : {col}', fontweight='bold')
    axes[i].set_ylabel('Valeur (BRL)')

plt.suptitle('Détection des Outliers (IQR Method)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantifier les outliers avec la méthode IQR
print('\nNombre d\'outliers détectés (méthode IQR):')
for col in cols_to_plot:
    Q1 = df_delivered[col].quantile(0.25)
    Q3 = df_delivered[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_delivered[(df_delivered[col] < Q1 - 1.5*IQR) | (df_delivered[col] > Q3 + 1.5*IQR)]
    print(f'  {col:25s}: {len(outliers):,} outliers ({len(outliers)/len(df_delivered)*100:.1f}%)')

---
## 6. Analyse des Variables Catégorielles

In [ ]:
# Top 15 catégories de produits par nombre de commandes
if 'product_category_name_english' in df_delivered.columns:
    top_categories = (df_delivered['product_category_name_english']
                      .value_counts().head(15))

    plt.figure(figsize=(12, 6))
    sns.barplot(x=top_categories.values, y=top_categories.index, palette='viridis')
    plt.title('Top 15 Catégories de Produits (par volume de commandes)', fontweight='bold')
    plt.xlabel('Nombre de commandes')
    plt.ylabel('Catégorie')
    plt.tight_layout()
    plt.show()

In [ ]:
# Risque return/refund par categorie de produit (Top 10)
if 'product_category_name_english' in df_delivered.columns:
    top10_cats = df_delivered['product_category_name_english'].value_counts().head(10).index
    df_top10 = df_delivered[df_delivered['product_category_name_english'].isin(top10_cats)]

    risk_by_cat = df_top10.groupby('product_category_name_english')['is_return_refund_risk'].mean().sort_values()

    plt.figure(figsize=(12, 5))
    plt.barh(risk_by_cat.index, risk_by_cat.values, color='salmon')
    plt.axvline(x=risk_by_cat.mean(), color='red', linestyle='--', label=f'Moyenne: {risk_by_cat.mean():.2f}')
    plt.xlabel('Taux de risque Return/Refund')
    plt.title('Risque Return/Refund par Categorie (Top 10)', fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution des modes de paiement
if 'payment_type' in df_delivered.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    payment_counts = df_delivered['payment_type'].value_counts()
    axes[0].pie(payment_counts.values, labels=payment_counts.index, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Repartition des Modes de Paiement', fontweight='bold')

    risk_by_payment = df_delivered.groupby('payment_type')['is_return_refund_risk'].mean().sort_values()
    risk_by_payment.plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_xlabel('Taux de risque Return/Refund')
    axes[1].set_title('Risque par Mode de Paiement', fontweight='bold')

    plt.tight_layout()
    plt.show()

---
## 7. Analyse Temporelle

Etude des tendances dans le temps : volumes de commandes et risque moyen par mois.

In [ ]:
df_delivered['order_purchase_timestamp'] = pd.to_datetime(
    df_delivered['order_purchase_timestamp'], errors='coerce'
)

df_delivered['year_month'] = df_delivered['order_purchase_timestamp'].dt.to_period('M')
monthly = df_delivered.groupby('year_month').agg(
    nb_commandes=('order_id', 'count'),
    taux_risque=('is_return_refund_risk', 'mean')
).reset_index()
monthly['year_month'] = monthly['year_month'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly['year_month'], monthly['nb_commandes'], marker='o', linewidth=2, color='steelblue')
axes[0].fill_between(range(len(monthly)), monthly['nb_commandes'], alpha=0.2, color='steelblue')
axes[0].set_ylabel('Nombre de commandes')
axes[0].set_title('Volume Mensuel de Commandes', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['year_month'], monthly['taux_risque'], marker='o', linewidth=2, color='salmon')
axes[1].axhline(y=monthly['taux_risque'].mean(), color='red', linestyle='--', label='Moyenne generale')
axes[1].set_ylabel('Taux de risque Return/Refund')
axes[1].set_title('Evolution Mensuelle du Risque Return/Refund', fontweight='bold')
axes[1].set_ylim(0.0, 0.6)
axes[1].legend()
plt.xticks(range(0, len(monthly), 2), monthly['year_month'][::2], rotation=45)

plt.tight_layout()
plt.show()

---
## 8. Analyse des Correlations

La heatmap de correlation montre les relations lineaires entre variables numeriques.
On inclut la cible proxy `is_return_refund_risk` pour identifier les signaux utiles.

In [ ]:
df_delivered['order_estimated_delivery_date'] = pd.to_datetime(
    df_delivered['order_estimated_delivery_date'], errors='coerce')
df_delivered['order_delivered_customer_date'] = pd.to_datetime(
    df_delivered['order_delivered_customer_date'], errors='coerce')
df_delivered['order_delivered_carrier_date'] = pd.to_datetime(
    df_delivered['order_delivered_carrier_date'], errors='coerce')

df_delivered['delivery_delay_days'] = (
    df_delivered['order_delivered_customer_date'] -
    df_delivered['order_estimated_delivery_date']
).dt.days

df_delivered['processing_days'] = (
    df_delivered['order_delivered_carrier_date'] -
    df_delivered['order_purchase_timestamp']
).dt.days

corr_cols = ['is_return_refund_risk', 'delivery_delay_days', 'processing_days',
             'total_price', 'total_freight', 'item_count',
             'payment_value', 'payment_installments', 'product_weight_g']
corr_cols = [c for c in corr_cols if c in df_delivered.columns]

corr_matrix = df_delivered[corr_cols].corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Matrice de Correlation des Variables Numeriques', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelations avec is_return_refund_risk (ordre decroissant):')
print(corr_matrix['is_return_refund_risk'].drop('is_return_refund_risk').sort_values(key=abs, ascending=False))

---
## 9. Conclusions de l'EDA

Synthèse des observations clés avant de passer au preprocessing.

### Points cles identifies

**Qualite des donnees :**
- Valeurs manquantes surtout dans `review_comment_message`, `product_category_name`, et certaines dates.
- Presence d'outliers sur prix et frais de port.
- Colonnes de dates a convertir avant modeling.

**Variable cible proxy :**
- Cible `is_return_refund_risk` desequilibree (majorite low risk).
- Utiliser `class_weight='balanced'` ou un ajustement de seuil.

**Features prometteuses :**
- `delivery_delay_days`
- `processing_days`
- `freight_ratio`
- signaux de paiement et categorie produit

**Tendances temporelles :**
- Variation du volume de commandes selon les mois.
- Variation du taux de risque selon les periodes.

**Prochaine etape :** `02_Modeling.ipynb` pour entrainer les versions finales;
les essais de parametres sont dans `brouillon_essai.ipynb`.